# Figure 5: personal functional profiles across intestinal sampling regions

CapScan, 347 LC-MS/MS acquisitions from 15 participants: four capsule sampling regions and stool.
Panel a is study-design artwork; panels b-g are regenerated here from `source_data/Fig5/` with the
original drawing code, checked pixel by pixel against the original render, and every Figure 5 statistic
in the Results is recomputed. Run from the repository root with `python paper.py build Fig5`.

Optional inputs: `FASTALAKE_FONT_DIR` (Söhne `.ttf` files, needed for pixel-exact text) and the three
Zenodo files of `source_data/Fig5/README.md` in `build/zenodo/Fig5/` (needed to recompute panels c and d
and the evenness test from acquisition-level profiles). Without them those steps print SKIPPED.

In [ ]:
from pathlib import Path
import sys, json
UNIT = Path.cwd()                 # the notebook runs with its own directory as cwd
REPO = UNIT.parents[1]
sys.path[:0] = [str(REPO), str(UNIT)]
SD = REPO / "source_data" / UNIT.name
OUT = REPO / "build" / UNIT.name   # rendered panels; gitignored
OUT.mkdir(parents=True, exist_ok=True)
from style import fl_style as fl   # imported per CONVENTIONS; fl.init() is not called: the Figure 5
import panels                      # producer did not use fl_style, and panels.setup_style() applies its exact settings
import numpy as np, pandas as pd
from IPython.display import Image, display
ZEN = REPO / "build" / "zenodo" / UNIT.name
HAVE_ZENODO = panels.zenodo_ready(ZEN)
print("fonts present:", panels.fonts_available(), "| Zenodo files present:", HAVE_ZENODO)
R = {}                            # every recomputed number, keyed as in panels.NUMBER_KEYS

## Panel a

Study design (artwork, `artwork/panel_a.pdf`, the producer's PDF unchanged). Supports "We analyzed 347
existing LC–MS/MS acquisitions from 308 specimens of fifteen healthy participants" (v28d line 147).
n = 347 acquisitions, 308 specimens, 15 participants. No test. The typed counts are compared with the
counts computed from `design_347.tsv`; every typed number is listed in `ARTWORK_NUMBERS.md`.

In [ ]:
import fitz
A = panels.stat_a_design(SD)
with fitz.open(UNIT / "artwork" / "panel_a.pdf") as doc:
    typed = doc[0].get_text()
    doc[0].get_pixmap(dpi=300, alpha=False).save(OUT / "panel_a.png")
want = [f"{A['acquisitions']} metaproteomes", f"{A['capsule']} capsule · {A['stool']} stool", f"· {A['participants']} participants"]
want += [f"{A['counts'][r]} acquisitions" for r in panels.REG]
missing = [w for w in want if w not in typed]
assert not missing, missing
print("typed in panel a and recomputed from design_347.tsv:", want)
n_matched = int(pd.read_csv(UNIT.parents[1] / "source_data" / "Fig4" / "crossfit_per_acquisition.tsv", sep="\t").cohort.eq("CAPSCAN").sum())
assert f"{n_matched} matched acquisitions" in typed, n_matched
print(f"'{n_matched} matched acquisitions' typed and recomputed from source_data/Fig4/crossfit_per_acquisition.tsv (CapScan rows)")
R.update(L147_acquisitions=A["acquisitions"], L147_specimens=A["specimens"], L147_participants=A["participants"],
         L147_capsule_regions_word=panels.as_word(A["capsule_regions"]))
display(Image(str(OUT / "panel_a.png"), width=700))

## Panel b

Mean human-assigned, microbial and unresolved shares of LFQ intensity per sampling group, with each
participant's human-assigned share. Supports "Regions 1, 2 and 3 had a higher human-assigned share than
stool ... Region 4 exceeded stool by a smaller 4.7 percentage points" (v28d line 149). n = 15
participants (75 participant-region cells from 346 acquisitions). Test: exact two-sided paired sign-flip
(32,768 patterns), percentile-bootstrap 95% CI (2,000 participant resamples), Benjamini-Hochberg over
four regions.

In [ ]:
cv = panels.Canvas(SD)
print(panels.panel_b(cv))
cells_new, tests_new = panels.stat_b_region_tests(SD)
cells_ref = pd.read_csv(SD / "origin_fraction_cells.tsv", sep="\t", float_precision="round_trip")
tests_ref = pd.read_csv(SD / "four_region_tests.tsv", sep="\t", float_precision="round_trip")
cols = ["Human-assigned", "Microbial above species", "Microbial species", "Unresolved origin"]
print("cells max |diff|:", panels.compare_tables(cells_new.reset_index(), cells_ref, ["participant", "region"], cols))
print("tests max |diff|:", panels.compare_tables(tests_new, tests_ref, ["region"], ["mean_region", "mean_stool", "mean_difference", "ci_low", "ci_high", "p", "q"]))
print(tests_new.to_string(index=False))
t = tests_new.set_index("region")
R.update(L149_region1_share=t.loc["duodenum", "mean_region"], L149_region4_share=t.loc["colon", "mean_region"],
         L149_stool_share=t.loc["duodenum", "mean_stool"], L149_participants=int(t.n.iloc[0]),
         L149_groups_word=panels.as_word(len(panels.REG)), L153_participants_b=int(t.n.iloc[0]))
for k, reg in zip(["r1", "r2", "r3", "r4"], panels.REG[:4]):
    R[f"L149_diff_{k}"] = t.loc[reg, "mean_difference"]; R[f"L149_ci_lo_{k}"] = t.loc[reg, "ci_low"]
    R[f"L149_ci_hi_{k}"] = t.loc[reg, "ci_high"]; R[f"L149_q_{k}"] = t.loc[reg, "q"]
STATS = {"b": tests_new}
display(cv.fig)

## Panel c

Mean distance from each participant's four capsule-region functional profiles to their own stool and
to the other fourteen stools. Supports "For each of the fifteen participants, the mean distance ... to
their own stool was smaller ... (mean difference 0.061, 95% CI 0.047–0.075, P≤1 × 10⁻⁵)" (v28d line
155). n = 15 participants, 75 participant-region profiles, 73,316 OGs. Test: two-sided permutation of
stool-to-participant assignments (99,999 permutations); percentile-bootstrap 95% CI (2,000 participant
resamples). The recomputation needs the Zenodo OG profile matrix.

In [ ]:
print(panels.panel_c(cv))
if HAVE_ZENODO:
    sim_new, own_new, part_new, roster_ok = panels.stat_cd_profiles(SD, ZEN)
    assert roster_ok, "og_profile_positive in design_347.tsv disagrees with the profile matrix"
    sim_ref = pd.read_csv(SD / "similarity_summary.tsv", sep="\t", float_precision="round_trip"); own_ref = pd.read_csv(SD / "own_stool_participants.tsv", sep="\t", float_precision="round_trip")
    part_ref = pd.read_csv(SD / "orthogonal_partition.tsv", sep="\t", float_precision="round_trip")
    print("similarity max |diff|:", panels.compare_tables(sim_new, sim_ref, ["prevalence"], ["features", "own_closer", "mean_own", "mean_other", "mean_advantage", "ci_low", "ci_high", "p"]))
    print("own-stool max |diff|:", panels.compare_tables(own_new, own_ref, ["prevalence", "participant"], ["own_stool_distance", "other_stool_mean_distance"]))
    print("partition max |diff|:", panels.compare_tables(part_new, part_ref, ["prevalence", "population", "component"], ["fraction"]))
    print(sim_new.to_string(index=False))
    s = sim_new.set_index("prevalence")
    R.update(L153_features=int(s.loc[0.0, "features"]), L155_closer=int(s.loc[0.0, "own_closer"]),
             L155_advantage=s.loc[0.0, "mean_advantage"], L155_ci_lo=s.loc[0.0, "ci_low"], L155_ci_hi=s.loc[0.0, "ci_high"],
             L155_p=s.loc[0.0, "p"], L281_features=int(s.loc[0.0, "features"]), L281_profiles=int(5 * s.loc[0.0, "participants"]),
             L847_profiles=int(5 * s.loc[0.0, "participants"]), L847_features=int(s.loc[0.0, "features"]),
             L847_features_10=int(s.loc[0.1, "features"]), L847_features_20=int(s.loc[0.2, "features"]),
             L847_p_sensitivity=float(max(s.loc[0.1, "p"], s.loc[0.2, "p"])))
    assert (s.own_closer == s.participants).all()
    STATS["c"] = sim_new
else:
    print("SKIPPED: panel c/d recomputation needs the Zenodo files in build/zenodo/Fig5 (see source_data/Fig5/README.md)")
display(cv.fig)

## Panel d

Share of functional-profile variation explained by participant, the capsule-stool contrast, differences
among capsule regions and residual, with stool (75 profiles) and capsules only (60 profiles). Supports
"participant identity accounted for 45.0% ... In the separate intestinal-fluid-only analysis,
participant identity accounted for 56.9% of variation and sampling region for 5.3%" (v28d line 155).
Descriptive sums-of-squares partition, no test. Recomputed together with panel c.

In [ ]:
print(panels.panel_d(cv))
if HAVE_ZENODO:
    pp = part_new[part_new.prevalence.eq(0.0)].set_index(["population", "component"]).fraction * 100
    a5, c4 = "All five regions", "Capsules only"
    R.update(L155_participant=pp[(a5, "Participant")], L155_sampling_group=pp[(a5, "Capsules versus stool")] + pp[(a5, "Between capsule regions")],
             L155_capsule_stool=pp[(a5, "Capsules versus stool")], L155_among_region=pp[(a5, "Between capsule regions")],
             L155_caps_participant=pp[(c4, "Participant")], L155_caps_region=pp[(c4, "Between capsule regions")],
             L849_capsule_stool_exact=pp[(a5, "Capsules versus stool")], L849_among_region_exact=pp[(a5, "Between capsule regions")],
             L849_caps_profiles=int(4 * s.loc[0.0, "participants"]), L849_caps_region_exact=pp[(c4, "Between capsule regions")],
             L849_sampling_group_exact=pp[(a5, "Capsules versus stool")] + pp[(a5, "Between capsule regions")])
    print(pp.round(4).to_string())
else:
    print("SKIPPED: needs the Zenodo files (see panel c)")
display(cv.fig)

## Panel e

Leading annotated species for 24 functions in each participant and sampling group (direct peptide
assignment). Supports "For the 24 displayed functions, the same annotated species led in 50.4% of
individual intestinal-fluid–own-stool comparisons (726 of 1,440) ... 111 of 360 participant–function
combinations (30.8%)" (v28d line 159; Methods line 787). n = 1,440 function-participant-region comparisons
and 360 participant-function profiles. Descriptive counts, no test.

In [ ]:
print(panels.panel_e(cv))
mr = pd.read_csv(SD / "mosaic_donor_regions.tsv", sep="\t")
kos = list(pd.read_csv(SD / "mosaic_abundance.tsv", sep="\t").sort_values("row").KO)
E = panels.stat_e_mosaic(mr, kos); print(E)
cov = panels.stat_species_label_coverage(SD); print(cov)
R.update(L153_functions=E["functions"], L157_functions=E["functions"], L159_functions=E["functions"],
         L159_same_leader_pct=100 * E["same_leader_fraction"], L159_same_leader=E["same_leader"], L159_comparisons=E["comparisons"],
         L159_persistence=E["persistence"], L159_persistence_profiles=E["persistence_profiles"],
         L159_persistence_pct=100 * E["persistence_fraction"], L787_persistence_pct=100 * E["persistence_fraction"],
         L159_label_coverage=100 * cov["median_resolved_fraction"])
display(cv.fig)

## Panel f

Per-participant agreement of the leading species between each capsule region and own stool under direct
peptide assignment (production only; the original protein grouping was removed in v30), for all eligible
functions and for the strict subset. Supports "the leading species in an intestinal sampling region matched
that in the participant's own stool in an average of 46.4% of comparisons ... increased agreement to 57.6%"
(v30 Results paragraph 81; v28d line 157 printed 45.6% and 60.0%). n = 15 participants. Descriptive means, no test.


In [ ]:
print(panels.panel_f(cv))
F = panels.stat_f_leaders(SD); print(F)
R.update(L157_resolved_direct=100 * F["resolved_direct"], L157_strict_direct=100 * F["strict_direct"],
         L157_participants=F["participants"])
display(cv.fig)


## Panel g

Spearman correlations between region-adjusted distance matrices of DNA, non-host protein, host protein
and metabolite profiles (native features, eggNOG OGs, KEGG KOs). Supports "distances between the adjusted
non-host protein profiles correlated with DNA, host protein and metabolite distances (ρ=0.51, 0.55 and
0.38, all q<0.001)" (v28d line 161). n = 13 participants, 65 participant-region profiles. Test: two-sided,
1,999 whole-participant block permutations; Benjamini-Hochberg over 6 native and over 16 functional
comparisons.

In [ ]:
print(panels.panel_g(cv))
nat_new = panels.stat_g_native(SD)
nat_ref = pd.read_csv(SD / "native_correlations.tsv", sep="\t", float_precision="round_trip"); nat_ref = nat_ref[nat_ref.analysis.eq(panels.NATIVE_ANALYSIS)]
print("native max |diff|:", panels.compare_tables(nat_new, nat_ref, ["layer_a", "layer_b"], ["rho", "p", "q"]))
fun_new = panels.stat_g_functional(SD)
fun_ref = pd.read_csv(SD / "functional_correlations.tsv", sep="\t", float_precision="round_trip")
fun_ref = fun_ref[fun_ref.scope.eq(panels.FUNCTION_SCOPE) & fun_ref.prevalence.eq(0.0)]
print("functional max |diff|:", panels.compare_tables(fun_new, fun_ref, ["level", "layer_a", "layer_b"], ["rho", "p", "q"]))
print(nat_new[["layer_a", "layer_b", "rho", "p", "q"]].to_string(index=False))
n_ = nat_new.set_index(["layer_a", "layer_b"])
R.update(L153_participants_g=int(nat_new.n_participants.iloc[0]), L161_participants=int(nat_new.n_participants.iloc[0]),
         L161_rho_dna=n_.loc[("DNA", "Non-host protein"), "rho"], L161_rho_host=n_.loc[("Non-host protein", "Host protein"), "rho"],
         L161_rho_met=n_.loc[("Non-host protein", "Metabolites"), "rho"], L161_q_dna=n_.loc[("DNA", "Non-host protein"), "q"],
         L161_q_host=n_.loc[("Non-host protein", "Host protein"), "q"], L161_q_met=n_.loc[("Non-host protein", "Metabolites"), "q"])
STATS.update(g_native=nat_new, g_functional=fun_new)
display(cv.fig)

## Text statistic: leading-species share and evenness

Not a panel. Supports "The leading species accounted for a median 22.5% of species-resolved signal in
intestinal fluid versus 12.7% in stool, and Pielou evenness was lower (0.623 versus 0.669; 13 of 15
participants in the expected direction for each measure; paired Wilcoxon P=6.1 × 10⁻⁴ and 3.1 × 10⁻⁴)"
(v28d line 159). n = 15 participants (paired). Test: exact two-sided Wilcoxon signed-rank on participant
medians over acquisitions. The per-acquisition metrics are recomputed from the Zenodo species matrix when
present, otherwise read from `evenness_per_acquisition.tsv`.

In [ ]:
acq_ref = pd.read_csv(SD / "evenness_per_acquisition.tsv", sep="\t", float_precision="round_trip")
if HAVE_ZENODO:
    acq = panels.stat_evenness_per_acquisition(SD, ZEN)
    print("per-acquisition max |diff|:", panels.compare_tables(acq, acq_ref, ["sample_id"], ["top1", "pielou", "richness", "resolved_mass", "species_resolved_fraction"]))
else:
    print("SKIPPED: per-acquisition recomputation needs function_species.npz; using evenness_per_acquisition.tsv")
    acq = acq_ref
ev, ev_part = panels.stat_evenness(acq)
ref = pd.read_csv(SD / "evenness_summary_tests.tsv", sep="\t", float_precision="round_trip")
print("summary max |diff|:", panels.compare_tables(ev, ref.drop(columns="source"), ["metric"],
      ["participant_median_fluid", "participant_median_stool", "wilcoxon_W", "wilcoxon_p_exact", "signflip_p_exact"]))
print(ev.to_string(index=False))
e = ev.set_index("metric")
assert e.loc["top1", "participants_fluid_gt_stool"] == e.loc["pielou", "participants_fluid_lt_stool"]
R.update(L159_top1_fluid=100 * e.loc["top1", "participant_median_fluid"], L159_top1_stool=100 * e.loc["top1", "participant_median_stool"],
         L159_pielou_fluid=e.loc["pielou", "participant_median_fluid"], L159_pielou_stool=e.loc["pielou", "participant_median_stool"],
         L159_expected_direction=int(e.loc["top1", "participants_fluid_gt_stool"]), L159_paired_n=int(e.loc["top1", "paired_participants"]),
         L159_wilcoxon_top1=e.loc["top1", "wilcoxon_p_exact"], L159_wilcoxon_pielou=e.loc["pielou", "wilcoxon_p_exact"])
STATS["evenness"] = ev

## Composite

Saves panels b-g as one PDF (the producer's `panels_bg_v33.pdf`), composes it with the panel-a artwork
(the producer's assembly, see `ASSEMBLY.md`), runs the producer's text checks, writes the plotted values
and one crop per panel (`panel_<x>.png`, boxes in `panels.PANEL_BOXES_PT`).

In [ ]:
panels.save_body(cv, OUT / "panels_bg.pdf")
info = panels.compose(UNIT / "artwork" / "panel_a.pdf", OUT / "panels_bg.pdf", OUT / "Fig5_v42.pdf", OUT / "Fig5_v42.png")
print(info, panels.check_composite(OUT / "Fig5_v42.pdf"))
panels.plotted_values(cv).to_csv(OUT / "Figure5_plotted_values.tsv", sep="\t", index=False)
crops = panels.crop_panels(OUT / "panels_bg.pdf", OUT)
print("inputs read by the panels:", json.dumps(cv.SRCS, indent=1))
display(Image(str(OUT / "Fig5_v42.png"), width=700))

## Checks

Pixel checks against the original producer's renders (`expected/`), then every recomputed number in
`NUMBERS.tsv` order, compared with the `recomputed_value` column.

In [ ]:
from tools.compare_values import same_numeric_value
import csv
from tools.check_pixels import compare
if panels.fonts_available():
    for ext in ("png", "pdf"):
        r = compare(OUT / f"Fig5_v42.{ext}", UNIT / "expected" / f"Fig5_v42.{ext}", UNIT / "checks" / f"Fig5_v42_{ext}", dpi=300)
        print(ext, r["verdict"], r.get("differing_pixels"))
        assert r["verdict"] == "IDENTICAL", r
else:
    print("pixel checks SKIPPED: fonts absent")

rows = list(csv.DictReader(open(UNIT / "NUMBERS.tsv", encoding="utf-8"), delimiter="\t"))
assert len(rows) == len(panels.NUMBER_KEYS), (len(rows), len(panels.NUMBER_KEYS))
bad = []
for key, row in zip(panels.NUMBER_KEYS, rows):
    if key not in R:
        print(f"{row['v28d_line']:>4}  {row['printed_value']:>12}  SKIPPED ({key})"); continue
    val = R[key]; val = val.item() if hasattr(val, "item") else val
    s = repr(val) if isinstance(val, float) else str(val)
    flag = "" if same_numeric_value(val, row["v42_recomputed_value"]) else "  <-- differs from NUMBERS.tsv"
    if flag: bad.append(key)
    print(f"{row['v28d_line']:>4}  {row['printed_value']:>12}  {s}{flag}")
json.dump({k: (v.item() if hasattr(v, "item") else v) for k, v in R.items()}, open(OUT / "recomputed.json", "w"), indent=1)
assert not bad, bad